In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [70]:
import os
from io import BytesIO

import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

In [71]:
def read_parquet_prefix(s3_client, bucket_name, prefix):
    """
    Read every Parquet object under a MinIO prefix
    and return one pandas DataFrame.
    """
    #MinIO may contain many objects. Instead of returning all objects in one response, the S3 API may divide them into pages.
    paginator = s3_client.get_paginator("list_objects_v2")

    #This list will store the names of all Parquet objects found.
    parquet_keys = []

    #This asks MinIO: Show me all objects inside this bucket whose key starts with this prefix.
    for page in paginator.paginate(Bucket=bucket_name,Prefix=prefix):
        #Contents contains the objects found by MinIO.
        for obj in page.get("Contents", []):
            #The key is the complete MinIO path of the file.
            key = obj["Key"]

            if key.endswith(".parquet"):
                parquet_keys.append(key)

    if not parquet_keys:
        raise FileNotFoundError(
            f"No Parquet files found under prefix: {prefix}"
        )

    dataframes = []

    for key in parquet_keys:
        response = s3_client.get_object(
            Bucket=bucket_name,
            Key=key
        )

        parquet_bytes = response["Body"].read()

        dataframe = pd.read_parquet(
            BytesIO(parquet_bytes),
            engine="fastparquet"
        )

        dataframes.append(dataframe)

        print(f"Loaded: {key}")
        print(f"Shape: {dataframe.shape}")

    combined_dataframe = pd.concat(
        dataframes,
        ignore_index=True
    )

    return combined_dataframe

def read_parquet_object(s3_client, bucket_name, object_key):
    response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    return pd.read_parquet(
        BytesIO(response["Body"].read()),
        engine="fastparquet",
    )

In [72]:
metadata_df = read_parquet_prefix(
    s3_client,
    BUCKET_NAME,
    "metadata/etl_metadata/"
)

required_silver_datasets = {"train_data", "test_data", "RUL_data"}
successful_silver_metadata = metadata_df[
    (metadata_df["layer"] == "silver")
    & (metadata_df["status"] == "success")
    & metadata_df["dataset_name"].isin(required_silver_datasets)
]

datasets_by_run = successful_silver_metadata.groupby("run_id")["dataset_name"].apply(set)
valid_silver_run_ids = sorted(
    run_id
    for run_id, datasets in datasets_by_run.items()
    if required_silver_datasets.issubset(datasets)
)

if not valid_silver_run_ids:
    raise FileNotFoundError("No successful Silver run contains train, test, and RUL data.")

pipeline_run_id = valid_silver_run_ids[-1]
selected_silver_metadata = successful_silver_metadata[
    successful_silver_metadata["run_id"] == pipeline_run_id
]
pipeline_load_date = selected_silver_metadata["load_date"].iloc[0]
metadata_key = (
    f"metadata/etl_metadata/load_date={pipeline_load_date}/"
    f"run_id={pipeline_run_id}/manifest.parquet"
)

def silver_object_key(dataset_name):
    matches = selected_silver_metadata[
        selected_silver_metadata["dataset_name"] == dataset_name
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Silver object for {dataset_name}.")
    return matches.iloc[0]["object_key"]

original_train_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("train_data")
)
original_test_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("test_data")
)
original_rul_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("RUL_data")
)

print(f"Selected Silver run: {pipeline_run_id}")

Loaded: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
Shape: (7, 8)
Selected Silver run: 20260902T120041719003Z


In [15]:
pd.set_option('display.max_columns', None)

In [73]:
train_df_minio = original_train_df_minio.copy()
test_df_minio = original_test_df_minio.copy()
rul_df_minio = original_rul_df_minio.copy()

In [ ]:
train_df_minio = train_df_minio.sort_values(
    ["unit_number", "time_in_cycles"]
).reset_index(drop=True)

In [74]:
train_df_minio = train_df_minio.drop(columns=["processing_timestamp", "data_quality_status"])
test_df_minio = test_df_minio.drop(columns=["processing_timestamp", "data_quality_status"])
rul_df_minio = rul_df_minio.drop(columns=["processing_timestamp", "data_quality_status"])

In [76]:
for column in train_df_minio.columns:
    if column not in ["unit_number","time_in_cycles","operational_setting_1","operational_setting_2","operational_setting_3"]:
        train_df_minio = train_df_minio.rename(columns={column: "current_" + column})

In [77]:
sensors = list(train_df_minio.columns) 
sensors.remove("unit_number")
sensors.remove("time_in_cycles")
sensors.remove("operational_setting_1")
sensors.remove("operational_setting_2")
sensors.remove("operational_setting_3")

In [ ]:
rolling_means = train_df_minio.groupby(["unit_number"])[sensors].rolling(window=20, min_periods=20).mean()

In [79]:
rolling_means = rolling_means.reset_index(level=0, drop=True)

In [81]:
for column in rolling_means.columns:
    if column == "unit_number":
        continue
    elif column == "level_1":
        rolling_means = rolling_means.rename(columns={column:"time_in_cycles"})
    else : 
        rolling_means = rolling_means.rename(columns={column: column[8:]+"_mean_20"})

In [82]:
train_df_minio = pd.concat([train_df_minio, rolling_means], axis=1)

In [ ]:
rolling_stds = train_df_minio.groupby("unit_number")[sensors].rolling(window=20, min_periods=20).std()

In [86]:
rolling_stds = rolling_stds.reset_index(level=0, drop=True)

In [89]:
for column in rolling_stds.columns:
    if column == "unit_number":
        continue
    else : 
        rolling_stds = rolling_stds.rename(columns={column: column[8:]+"_std_20"})

In [91]:
train_df_minio = pd.concat([train_df_minio,rolling_stds],axis=1)

In [ ]:
rolling_min = train_df_minio.groupby("unit_number")[sensors].rolling(window=20, min_periods=20).min()
rolling_min = rolling_min.reset_index(level=0, drop=True)
for column in rolling_min.columns:
    if column == "unit_number":
        continue
    else : 
        rolling_min = rolling_min.rename(columns={column: column[8:]+"_min_20"})
train_df_minio = pd.concat([train_df_minio,rolling_min],axis=1)

In [ ]:
rolling_max = train_df_minio.groupby("unit_number")[sensors].rolling(window=20, min_periods=20).max()
rolling_max = rolling_max.reset_index(level=0, drop=True)
for column in rolling_max.columns:
    if column == "unit_number":
        continue
    else : 
        rolling_max = rolling_max.rename(columns={column: column[8:]+"_max_20"})
train_df_minio = pd.concat([train_df_minio,rolling_max],axis=1)

In [99]:
difference = train_df_minio.groupby("unit_number")[sensors].diff()

In [109]:
for column in difference.columns:
    difference = difference.rename(columns={column: column[8:]+"_difference"})
train_df_minio = pd.concat([train_df_minio,difference],axis=1)

In [117]:
percentage_change = (
    train_df_minio
    .groupby("unit_number")[sensors]
    .pct_change(fill_method=None)
    .replace([np.inf, -np.inf], np.nan)
)
for column in percentage_change.columns:
    percentage_change = percentage_change.rename(columns={column: column[8:]+f"_%change"})
train_df_minio = pd.concat([train_df_minio,percentage_change],axis=1)

In [ ]:
sensor_slope = train_df_minio.groupby("unit_number")[sensors].rolling(window=20, min_periods=20).apply(lambda x: np.polyfit(np.arange(len(x)),x,1)[0], raw=True).reset_index(level=0,drop=True)

In [127]:
for column in sensor_slope.columns:
    sensor_slope = sensor_slope.rename(columns={column: column[8:]+f"_sensor_slope_20"})
train_df_minio = pd.concat([train_df_minio,sensor_slope],axis=1)

In [ ]:
ema_features = (
    train_df_minio
    .groupby("unit_number")[sensors]
    .transform(
        lambda column: column.ewm(
            span=20,
            adjust=False,
            min_periods=1
        ).mean()
    )
)

ema_features = ema_features.rename(
    columns={
        column: column[8:] + "_ema_20"
        for column in ema_features.columns
    }
)

train_df_minio = pd.concat(
    [train_df_minio, ema_features],
    axis=1
)

In [31]:
train_df_minio.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,current_sensor_measurement_1,current_sensor_measurement_2,current_sensor_measurement_3,current_sensor_measurement_4,current_sensor_measurement_5,current_sensor_measurement_6,current_sensor_measurement_7,current_sensor_measurement_8,current_sensor_measurement_9,current_sensor_measurement_10,current_sensor_measurement_11,current_sensor_measurement_12,current_sensor_measurement_13,current_sensor_measurement_14,current_sensor_measurement_15,current_sensor_measurement_16,current_sensor_measurement_17,current_sensor_measurement_18,current_sensor_measurement_19,current_sensor_measurement_20,current_sensor_measurement_21,sensor_measurement_1_mean_20,sensor_measurement_2_mean_20,sensor_measurement_3_mean_20,sensor_measurement_4_mean_20,sensor_measurement_5_mean_20,sensor_measurement_6_mean_20,sensor_measurement_7_mean_20,sensor_measurement_8_mean_20,sensor_measurement_9_mean_20,sensor_measurement_10_mean_20,sensor_measurement_11_mean_20,sensor_measurement_12_mean_20,sensor_measurement_13_mean_20,sensor_measurement_14_mean_20,sensor_measurement_15_mean_20,sensor_measurement_16_mean_20,sensor_measurement_17_mean_20,sensor_measurement_18_mean_20,sensor_measurement_19_mean_20,sensor_measurement_20_mean_20,sensor_measurement_21_mean_20,sensor_measurement_1_std_20,sensor_measurement_2_std_20,sensor_measurement_3_std_20,sensor_measurement_4_std_20,sensor_measurement_5_std_20,sensor_measurement_6_std_20,sensor_measurement_7_std_20,sensor_measurement_8_std_20,sensor_measurement_9_std_20,sensor_measurement_10_std_20,sensor_measurement_11_std_20,sensor_measurement_12_std_20,sensor_measurement_13_std_20,sensor_measurement_14_std_20,sensor_measurement_15_std_20,sensor_measurement_16_std_20,sensor_measurement_17_std_20,sensor_measurement_18_std_20,sensor_measurement_19_std_20,sensor_measurement_20_std_20,sensor_measurement_21_std_20,sensor_measurement_1_min_20,sensor_measurement_2_min_20,sensor_measurement_3_min_20,sensor_measurement_4_min_20,sensor_measurement_5_min_20,sensor_measurement_6_min_20,sensor_measurement_7_min_20,sensor_measurement_8_min_20,sensor_measurement_9_min_20,sensor_measurement_10_min_20,sensor_measurement_11_min_20,sensor_measurement_12_min_20,sensor_measurement_13_min_20,sensor_measurement_14_min_20,sensor_measurement_15_min_20,sensor_measurement_16_min_20,sensor_measurement_17_min_20,sensor_measurement_18_min_20,sensor_measurement_19_min_20,sensor_measurement_20_min_20,sensor_measurement_21_min_20,sensor_measurement_1_max_20,sensor_measurement_2_max_20,sensor_measurement_3_max_20,sensor_measurement_4_max_20,sensor_measurement_5_max_20,sensor_measurement_6_max_20,sensor_measurement_7_max_20,sensor_measurement_8_max_20,sensor_measurement_9_max_20,sensor_measurement_10_max_20,sensor_measurement_11_max_20,sensor_measurement_12_max_20,sensor_measurement_13_max_20,sensor_measurement_14_max_20,sensor_measurement_15_max_20,sensor_measurement_16_max_20,sensor_measurement_17_max_20,sensor_measurement_18_max_20,sensor_measurement_19_max_20,sensor_measurement_20_max_20,sensor_measurement_21_max_20,sensor_measurement_1_difference,sensor_measurement_2_difference,sensor_measurement_3_difference,sensor_measurement_4_difference,sensor_measurement_5_difference,sensor_measurement_6_difference,sensor_measurement_7_difference,sensor_measurement_8_difference,sensor_measurement_9_difference,sensor_measurement_10_difference,sensor_measurement_11_difference,sensor_measurement_12_difference,sensor_measurement_13_difference,sensor_measurement_14_difference,sensor_measurement_15_difference,sensor_measurement_16_difference,sensor_measurement_17_difference,sensor_measurement_18_difference,sensor_measurement_19_difference,sensor_measurement_20_difference,sensor_measurement_21_difference,sensor_measurement_1_%change,sensor_measurement_2_%change,sensor_measurement_3_%change,sensor_measurement_4_%change,sensor_measurement_5_%change,sensor_measurement_6_%change,sensor

In [32]:
test_df_minio.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,sensor_measurement_6,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_10,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_14,sensor_measurement_15,sensor_measurement_16,sensor_measurement_17,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,21.61,553.90,2388.04,9050.17,1.3,47.20,521.72,2388.03,8125.55,8.4052,0.03,392.0,2388.0,100.0,38.86,23.3735
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,21.61,554.85,2388.01,9054.42,1.3,47.50,522.16,2388.06,8139.62,8.3803,0.03,393.0,2388.0,100.0,39.02,23.3916
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,21.61,554.11,2388.05,9056.96,1.3,47.50,521.97,2388.03,8130.10,8.4441,0.03,393.0,2388.0,100.0,39.08,23.4166
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,21.61,554.07,2388.03,9045.29,1.3,47.28,521.38,2388.05,8132.90,8.3917,0.03,391.0,2388.0,100.0,39.00,23.3737
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,21.61,554.16,2388.01,9044.55,1.3,47.31,522.15,2388.03,8129.54,8.4031,0.03,390.0,2388.0,100.0,38.99,23.4130
